# Token SAE Training on PACE (ICE Cluster)

Train the token-based Sparse Autoencoder on PACE on-demand. Set your protein directory path and **latent dimensions** below, then run all cells.

**Data structure expected:**
```
protein_dir/
├── 6tf4_A/
│   └── 6tf4_A_pair_block_47.npy   # Shape: (L, L, 128)
├── 7xyz_B/
│   └── 7xyz_B_pair_block_47.npy
...
```

## 1. Configuration (edit latent dimensions here)

In [ ]:
# ============ ARCHITECTURE & TRAINING CONFIG ============
# Change d_latent to control SAE capacity (expansion factor = d_latent / 128)

D_LATENT = 3000          # Latent dimensions (e.g. 1024, 3000, 4096, 8192, 16384)
TAU = 0.90              # CDF threshold for adaptive top-k
LAMBDA_ENTROPY = 0.01   # Entropy penalty weight

# Training hyperparameters
EPOCHS = 100
BATCH_SIZE = 16
LR = 1e-3
VAL_FRAC = 0.2
SEED = 42

# Paths (on PACE, use your scratch path)
# Option A: Set via env on cluster: export BASE=/path/to/CompleteProteins
# Option B: Set directly here
import os
PROTEIN_DIR = os.environ.get("BASE", os.environ.get("PROTEIN_DIR", "/path/to/CompleteProteins"))
OUTPUT_DIR = None  # None = auto: protein_dir/token_sae_output

LAYER = 47  # Pair block layer from the model

print(f"Latent dimensions: {D_LATENT} (expansion: {D_LATENT/128:.1f}x)")
print(f"Protein dir: {PROTEIN_DIR}")
print(f"Output: {OUTPUT_DIR or f'{PROTEIN_DIR}/token_sae_output'}")

## 2. Setup: imports and data discovery

In [ ]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

SCRIPT_DIR = Path(".").resolve()
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

from sparse_autoencoder import (
    PairRepresentationDataset,
    TokenSparseAutoencoder,
    pack_proteins_collate,
)


def discover_pair_paths(protein_dir: Path, layer: int = 47):
    """Return list of paths to *_pair_block_{layer}.npy under protein_dir."""
    paths = []
    if not protein_dir.is_dir():
        return paths
    for item in sorted(protein_dir.iterdir()):
        if not item.is_dir():
            continue
        name = item.name
        p = item / f"{name}_pair_block_{layer}.npy"
        if p.is_file():
            paths.append(str(p))
    return paths


protein_dir = Path(PROTEIN_DIR)
paths = discover_pair_paths(protein_dir, LAYER)

if not paths:
    raise FileNotFoundError(
        f"No *_pair_block_{LAYER}.npy found under {protein_dir}. "
        "Check PROTEIN_DIR and data layout."
    )

print(f"Found {len(paths)} proteins")
print(f"Example: {paths[0]}")

## 3. Create dataset and model

In [ ]:
torch.manual_seed(SEED)

full_dataset = PairRepresentationDataset(paths, normalize=True)
n_val = max(1, int(len(full_dataset) * VAL_FRAC))
n_train = len(full_dataset) - n_val
train_dataset, val_dataset = random_split(full_dataset, [n_train, n_val])

NUM_WORKERS = 4 if torch.cuda.is_available() else 0
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=pack_proteins_collate,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=pack_proteins_collate,
    num_workers=NUM_WORKERS,
    persistent_workers=NUM_WORKERS > 0,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TokenSparseAutoencoder(
    d_in=128,
    d_latent=D_LATENT,
    tau=TAU,
).to(device)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

output_dir = Path(OUTPUT_DIR or str(protein_dir / "token_sae_output"))
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Device: {device} ({torch.cuda.device_count()} GPU(s))")
print(f"Train samples: {n_train}, Val samples: {n_val}")
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

## 4. Training loop

In [ ]:
def train_epoch(model, dataloader, optimizer, device, lambda_entropy: float):
    model.train()
    total_loss = total_recon = total_entropy = 0.0
    num_batches = 0
    for packed_batch, _ in dataloader:
        packed_batch = packed_batch.to(device, non_blocking=True)
        optimizer.zero_grad()
        recon_packed, p_softmax, _ = model(packed_batch)
        loss_recon = nn.functional.mse_loss(recon_packed, packed_batch)
        entropy = -(p_softmax * (p_softmax + 1e-10).log()).sum(dim=-1).mean()
        loss_total = loss_recon + lambda_entropy * entropy
        loss_total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss_total.item()
        total_recon += loss_recon.item()
        total_entropy += entropy.item()
        num_batches += 1
    n = max(num_batches, 1)
    return total_loss / n, total_recon / n, total_entropy / n


@torch.no_grad()
def evaluate(model, dataloader, device, lambda_entropy: float):
    model.eval()
    total_loss = total_recon = total_entropy = 0.0
    num_batches = 0
    for packed_batch, _ in dataloader:
        packed_batch = packed_batch.to(device, non_blocking=True)
        recon_packed, p_softmax, _ = model(packed_batch)
        loss_recon = nn.functional.mse_loss(recon_packed, packed_batch)
        entropy = -(p_softmax * (p_softmax + 1e-10).log()).sum(dim=-1).mean()
        loss_total = loss_recon + lambda_entropy * entropy
        total_loss += loss_total.item()
        total_recon += loss_recon.item()
        total_entropy += entropy.item()
        num_batches += 1
    n = max(num_batches, 1)
    return total_loss / n, total_recon / n, total_entropy / n

In [ ]:
best_val_loss = float("inf")

for epoch in range(EPOCHS):
    train_loss, train_recon, train_entropy = train_epoch(
        model, train_loader, optimizer, device, LAMBDA_ENTROPY
    )
    val_loss, val_recon, val_entropy = evaluate(
        model, val_loader, device, LAMBDA_ENTROPY
    )

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train: {train_loss:.6f} (recon: {train_recon:.6f}, ent: {train_entropy:.6f}) | "
        f"Val: {val_loss:.6f} (recon: {val_recon:.6f}, ent: {val_entropy:.6f})"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
        torch.save(state, output_dir / "token_sae_best.pt")

state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
torch.save(state, output_dir / "token_sae_final.pt")
print(f"\nSaved checkpoints to {output_dir}")